In [1]:
"""
Dextramer counts CD200high vs Others - Python conversion from R
Generates multi-page PDF plots comparing clonotype responses across clusters.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import os

# Set working directory to script location (equivalent to R's setwd)
# In Python, you'd typically run from the directory containing your data
# or specify the path explicitly
# os.chdir(os.path.dirname(os.path.abspath(__file__)))

# ------------------------------ Dextramer counts CD200high vs Others ------------------------------

# Read data
melted_df = pd.read_csv("../Data/20260116 Comparison 3/20260122 example input.csv")

# Drop unnamed index column if present (equivalent to melted_df$X <- NULL)
if 'Unnamed: 0' in melted_df.columns:
    melted_df = melted_df.drop(columns=['Unnamed: 0'])
if 'X' in melted_df.columns:
    melted_df = melted_df.drop(columns=['X'])

print(melted_df)

# Reorder levels (these would be used for factor ordering)
peptide_levels = ["EEEPVKKI", "ATLVFHNL", "HIYEFPQL", "INFDFPKL", "RAYLFNSV", "RTYTYEKL",
                  "SNYLFTKL", "SSYTFPKM", "SVYVYKVL", "VAFDFTKV", "VGPRYTNL", "VIVRFLTV", "VSFTYRYL"]
comp_levels = ["All", "CD200high"]  # Change to whatever we are comparing

# Color mapping
color_map = {"All": "#B3B3B3", "CD200high": "#f9a034"}  # grey70 ≈ #B3B3B3

# Set up iterative plotting
clonotypes = melted_df['Clonotype'].unique()
plot_list = []


def create_clonotype_plot(plot_data, ct, ax):
    """Create a single clonotype bar plot with error bars."""
    if len(plot_data) == 0:
        ax.axis('off')
        return
    
    # Get unique clusters and x positions
    clusters = plot_data['Cluster'].unique()
    x_values = plot_data['column_name'].unique()
    
    # Sort x_values if they're in peptide_levels
    x_values = sorted(x_values, key=lambda x: peptide_levels.index(x) if x in peptide_levels else float('inf'))
    
    x_positions = np.arange(len(x_values))
    width = 0.35
    n_clusters = len(clusters)
    
    # Calculate offsets for dodging
    if n_clusters == 1:
        offsets = [0]
    else:
        offsets = np.linspace(-width/2, width/2, n_clusters)
    
    # Plot bars for each cluster
    for i, cluster in enumerate(comp_levels):  # Use comp_levels to ensure consistent ordering
        if cluster not in clusters:
            continue
        cluster_data = plot_data[plot_data['Cluster'] == cluster]
        
        # Match data to x positions
        means = []
        sems = []
        for x_val in x_values:
            row = cluster_data[cluster_data['column_name'] == x_val]
            if len(row) > 0:
                means.append(row['mean'].values[0])
                sems.append(row['sem'].values[0] if 'sem' in row.columns else 0)
            else:
                means.append(0)
                sems.append(0)
        
        color = color_map.get(cluster, '#888888')
        ax.bar(x_positions + offsets[i] if n_clusters > 1 else x_positions, 
               means, width=width * 0.85, 
               label=cluster, color=color, edgecolor='none')
        ax.errorbar(x_positions + offsets[i] if n_clusters > 1 else x_positions, 
                    means, yerr=sems, 
                    fmt='none', color='black', capsize=2, linewidth=1)
    
    # Get counts per cluster for annotation
    cluster_n = plot_data.drop_duplicates(subset=['Cluster', 'CTCount'])[['Cluster', 'CTCount']]
    cluster_n = cluster_n.sort_values('Cluster')
    
    # Add count annotations in top right
    y_start = 20
    y_step = 1.2
    for idx, (_, row) in enumerate(cluster_n.iterrows()):
        y_pos = y_start - idx * y_step
        ax.text(0.98, y_pos / 20,  # Normalize to axes coordinates
                f"{row['Cluster']}: n = {int(row['CTCount'])}", 
                transform=ax.transAxes,
                ha='right', va='top', fontsize=8)
    
    # Format title (add newline before _BV)
    title = ct.replace('_BV', '\n_BV')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('Mean Count', fontsize=9)
    ax.set_ylim(0, 20)  # Change this for y axis scaling
    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_values, rotation=90, ha='center', fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


# Define plots per page
plots_per_page = 32
ncol = 4
nrow = plots_per_page // ncol

total_pages = int(np.ceil(len(clonotypes) / plots_per_page))

# Loop through clonotypes and save each page
for page in range(total_pages):
    start_index = page * plots_per_page
    end_index = min((page + 1) * plots_per_page, len(clonotypes))
    current_clonotypes = clonotypes[start_index:end_index]
    
    # Create figure
    fig, axes = plt.subplots(nrow, ncol, figsize=(18, 30))
    axes = axes.flatten()
    
    # Create plots
    for idx, ct in enumerate(current_clonotypes):
        plot_data = melted_df[melted_df['Clonotype'] == ct].copy()
        create_clonotype_plot(plot_data, ct, axes[idx])
    
    # Clear unused axes
    for idx in range(len(current_clonotypes), plots_per_page):
        axes[idx].axis('off')
    
    plt.tight_layout()
    
    # Save to PDF
    output_filename = f"Example Output_{page + 1:02d}.pdf"
    plt.savefig(output_filename, format='pdf', bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_filename}")

print("Done!")

                                              Clonotype    Cluster  \
0     AV16-CAMRESNTGYQNFYF-AJ49_BV13-2-CASGTGGQNTLYF...        All   
1     AV16-CAMRESNTGYQNFYF-AJ49_BV13-2-CASGTGGQNTLYF...        All   
2     AV16-CAMRESNTGYQNFYF-AJ49_BV13-2-CASGTGGQNTLYF...        All   
3     AV16-CAMRESNTGYQNFYF-AJ49_BV13-2-CASGTGGQNTLYF...        All   
4     AV16-CAMRESNTGYQNFYF-AJ49_BV13-2-CASGTGGQNTLYF...        All   
...                                                 ...        ...   
1139  AV2-CIVTDMNYNQGKLIF-AJ23_BV19-CASSPTGPNSDYTF-B...  CD200high   
1140  AV2-CIVTDMNYNQGKLIF-AJ23_BV19-CASSPTGPNSDYTF-B...  CD200high   
1141  AV2-CIVTDMNYNQGKLIF-AJ23_BV19-CASSPTGPNSDYTF-B...  CD200high   
1142  AV2-CIVTDMNYNQGKLIF-AJ23_BV19-CASSPTGPNSDYTF-B...  CD200high   
1143  AV2-CIVTDMNYNQGKLIF-AJ23_BV19-CASSPTGPNSDYTF-B...  CD200high   

     column_name  CTCount      mean       sem  
0       ATLVFHNL      414  1.224638  0.144053  
1       EEEPVKKI      414  0.524155  0.087238  
2       HIYEFPQ